In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import catboost as cb
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import gc
import warnings

warnings.filterwarnings('ignore')

print("--- 1. Loading Data ---")

def reduce_mem_usage(df, verbose=True):
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and col_type.name != 'category' and 'datetime' not in str(col_type):
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max: df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max: df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max: df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max: df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max: df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max: df[col] = df[col].astype(np.float32)
                else: df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose: print(f'Memory usage reduced to {end_mem:.2f} MB ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    return df

try:
    train_df = reduce_mem_usage(pd.read_csv('/kaggle/input/trainn/train_data.csv'))
    test_df = reduce_mem_usage(pd.read_csv('/kaggle/input/test-data/test_data.csv'))
    add_trans_df = reduce_mem_usage(pd.read_csv('/kaggle/input/transs1/add_trans.csv'))
    add_event_df = reduce_mem_usage(pd.read_csv('/kaggle/input/eventt/event_data_full.csv'))
    offer_metadata_df = reduce_mem_usage(pd.read_csv('/kaggle/input/metaaa/offer_metadata.csv'))
    print("All datasets loaded and memory optimized successfully.")
except FileNotFoundError as e:
    print(f"Error loading data: {e}.")
    exit()


In [ ]:
print("\n--- 2. Advanced Feature Engineering ---")

def feature_engineer_advanced(df, trans_df, event_df, offer_df):
    # --- Base Features ---
    df['id4'] = pd.to_datetime(df['id4'], errors='coerce')
    df['impression_day_of_week'] = df['id4'].dt.dayofweek
    df['impression_hour'] = df['id4'].dt.hour
    
    offer_df['id12'] = pd.to_datetime(offer_df['id12'], errors='coerce')
    offer_df['id13'] = pd.to_datetime(offer_df['id13'], errors='coerce')
    offer_df['offer_duration_days'] = (offer_df['id13'] - offer_df['id12']).dt.days
    df = pd.merge(df, offer_df[['id3', 'offer_duration_days', 'id12']], on='id3', how='left')
    df['days_since_offer_launch'] = (df['id4'] - df['id12']).dt.days

    # --- Time-Series & Sequential Features ---
    df = df.sort_values(by=['id2', 'id4']).reset_index(drop=True)
    df['time_since_last_impression'] = df.groupby('id2')['id4'].diff().dt.total_seconds()
    df['customer_offer_rank'] = df.groupby('id2').cumcount() + 1
    
    # --- NEW: Session-based features ---
    # A session is defined as a series of impressions with less than 30 minutes between them
    df['session_id'] = (df['time_since_last_impression'] > 1800).cumsum()
    session_agg = df.groupby(['id2', 'session_id'])['id4'].agg(['count', 'min', 'max']).reset_index()
    session_agg['time_in_session'] = (session_agg['max'] - session_agg['min']).dt.total_seconds()
    session_agg.rename(columns={'count': 'offers_in_session'}, inplace=True)
    df = pd.merge(df, session_agg[['id2', 'session_id', 'offers_in_session', 'time_in_session']], on=['id2', 'session_id'], how='left')

    # --- NEW: Time-Decay Weighted Transaction Aggregates ---
    print("  - Creating time-decay weighted transaction features...")
    trans_df['f370'] = pd.to_datetime(trans_df['f370'], errors='coerce')
    trans_df = trans_df.sort_values(by='f370')
    # Calculate days from the most recent transaction
    trans_df['days_ago'] = (trans_df['f370'].max() - trans_df['f370']).dt.days
    # Apply exponential decay weight
    trans_df['decay_weight'] = np.exp(-0.1 * trans_df['days_ago'])
    trans_df['weighted_amount'] = trans_df['f367'] * trans_df['decay_weight']
    
    time_decay_agg = trans_df.groupby('id2').agg(
        weighted_mean_trans=('weighted_amount', 'mean'),
        trans_count_last_7_days=('days_ago', lambda x: (x <= 7).sum())
    ).reset_index()
    df = pd.merge(df, time_decay_agg, on='id2', how='left')
    del time_decay_agg
    gc.collect()

    # --- NEW: Historical CTR with Bayesian Smoothing ---
    print("  - Creating smoothed historical CTRs...")
    # Calculate global CTR for smoothing
    global_ctr = event_df['id7'].notna().mean()
    # Customer CTR
    customer_agg = event_df.groupby('id2').agg(total_impressions=('id4', 'count'), total_clicks=('id7', lambda x: x.notna().sum())).reset_index()
    # Smoothing factor (e.g., 20)
    C = 20
    customer_agg['customer_ctr_smoothed'] = (customer_agg['total_clicks'] + C * global_ctr) / (customer_agg['total_impressions'] + C)
    df = pd.merge(df, customer_agg[['id2', 'customer_ctr_smoothed']], on='id2', how='left')
    del customer_agg
    gc.collect()

    # --- NEW: Offer Popularity Trend ---
    print("  - Creating offer popularity trend features...")
    event_df['impression_date'] = pd.to_datetime(event_df['id4']).dt.date
    offer_popularity = event_df.groupby(['id3', 'impression_date']).size().reset_index(name='impressions_per_day')
    offer_popularity = offer_popularity.groupby('id3')['impressions_per_day'].mean().reset_index()
    df = pd.merge(df, offer_popularity, on='id3', how='left')
    del offer_popularity
    gc.collect()
    
    return df

train_df = feature_engineer_advanced(train_df, add_trans_df, add_event_df, offer_metadata_df)
test_df = feature_engineer_advanced(test_df, add_trans_df, add_event_df, offer_metadata_df)
del add_trans_df, add_event_df
gc.collect()

print("\n--- 3. Data Preparation for Models ---")
train_df = pd.merge(train_df, offer_metadata_df, on='id3', how='left', suffixes=('', '_meta'))
test_df = pd.merge(test_df, offer_metadata_df, on='id3', how='left', suffixes=('', '_meta'))
del offer_metadata_df
gc.collect()

features = [col for col in train_df.columns if col not in ['id1', 'id2', 'id4', 'id5', 'y', 'id12', 'id13', 'id12_meta', 'id13_meta', 'session_id']]
features = [f for f in features if f in test_df.columns]
categorical_features = train_df[features].select_dtypes(include=['object', 'category']).columns.tolist()

X_cat_xgb = train_df[features].copy()
X_test_cat_xgb = test_df[features].copy()
for col in categorical_features:
    X_cat_xgb[col] = X_cat_xgb[col].astype(str).fillna('missing')
    X_test_cat_xgb[col] = X_test_cat_xgb[col].astype(str).fillna('missing')
numerical_features = X_cat_xgb.select_dtypes(include=np.number).columns.tolist()
for col in numerical_features:
    X_cat_xgb[col].fillna(-999, inplace=True)
    X_test_cat_xgb[col].fillna(-999, inplace=True)

X_lgbm = train_df[features].copy()
X_test_lgbm = test_df[features].copy()
for col in categorical_features:
    le = LabelEncoder()
    combined = pd.concat([X_lgbm[col], X_test_lgbm[col]], axis=0).astype(str)
    le.fit(combined)
    X_lgbm[col] = le.transform(X_lgbm[col].astype(str))
    X_test_lgbm[col] = le.transform(X_test_lgbm[col].astype(str))
X_lgbm.fillna(-999, inplace=True)
X_test_lgbm.fillna(-999, inplace=True)

y = train_df['y']


In [ ]:
lgbm_best_params = {'learning_rate': 0.03706439110944439, 'num_leaves': 46, 'max_depth': 9, 'colsample_bytree': 0.6111965410620912}
cat_best_params = {'learning_rate': 0.08213064681305168, 'depth': 10, 'l2_leaf_reg': 5.423793926976912}
xgb_best_params = {'learning_rate': 0.0310491750215814, 'max_depth': 9, 'subsample': 0.8499766666830204, 'colsample_bytree': 0.7918851587557293}

print(f"Using LGBM Params: {lgbm_best_params}")
print(f"Using CatBoost Params: {cat_best_params}")
print(f"Using XGBoost Params: {xgb_best_params}")


In [ ]:
from sklearn.model_selection import StratifiedKFold


In [ ]:
print("\n--- 5. Stacking Ensemble Training ---")
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_preds_lgbm = np.zeros(len(train_df))
oof_preds_cat = np.zeros(len(train_df))
oof_preds_xgb = np.zeros(len(train_df))
test_preds_lgbm = np.zeros(len(test_df))
test_preds_cat = np.zeros(len(test_df))
test_preds_xgb = np.zeros(len(test_df))

lgbm_params = {**lgbm_best_params, 'objective': 'binary', 'metric': 'average_precision', 'device': 'gpu', 'n_estimators': 2000, 'seed': 42, 'n_jobs': -1, 'verbose': -1}
cat_params = {**cat_best_params, 'iterations': 2500, 'eval_metric': 'AUC', 'task_type': 'GPU', 'random_seed': 42, 'verbose': 0}
xgb_params = {**xgb_best_params, 'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'tree_method': 'hist', 'device': 'cuda', 'n_estimators': 2000, 'seed': 42, 'n_jobs': -1}

for fold, (train_idx, val_idx) in enumerate(skf.split(X_lgbm, y)):
    print(f"--- Training Fold {fold+1}/{N_SPLITS} ---")
    lgbm = lgb.LGBMClassifier(**lgbm_params)
    lgbm.fit(X_lgbm.iloc[train_idx], y.iloc[train_idx], eval_set=[(X_lgbm.iloc[val_idx], y.iloc[val_idx])], callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_preds_lgbm[val_idx] = lgbm.predict_proba(X_lgbm.iloc[val_idx])[:, 1]
    test_preds_lgbm += lgbm.predict_proba(X_test_lgbm)[:, 1] / N_SPLITS
    
    cat = cb.CatBoostClassifier(**cat_params)
    cat.fit(X_cat_xgb.iloc[train_idx], y.iloc[train_idx], eval_set=(X_cat_xgb.iloc[val_idx], y.iloc[val_idx]), cat_features=categorical_features, early_stopping_rounds=100, use_best_model=True)
    oof_preds_cat[val_idx] = cat.predict_proba(X_cat_xgb.iloc[val_idx])[:, 1]
    test_preds_cat += cat.predict_proba(X_test_cat_xgb)[:, 1] / N_SPLITS
    
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(X_lgbm.iloc[train_idx], y.iloc[train_idx], eval_set=[(X_lgbm.iloc[val_idx], y.iloc[val_idx])], early_stopping_rounds=100, verbose=False)
    oof_preds_xgb[val_idx] = xgb_model.predict_proba(X_lgbm.iloc[val_idx])[:, 1]
    test_preds_xgb += xgb_model.predict_proba(X_test_lgbm)[:, 1] / N_SPLITS

print("\n--- Training Meta-Model ---")
stack_train = pd.DataFrame({'lgbm': oof_preds_lgbm, 'cat': oof_preds_cat, 'xgb': oof_preds_xgb})
stack_test = pd.DataFrame({'lgbm': test_preds_lgbm, 'cat': test_preds_cat, 'xgb': test_preds_xgb})
meta_model = LogisticRegression(random_state=42)
meta_model.fit(stack_train, y)
final_predictions = meta_model.predict_proba(stack_test)[:, 1]


In [ ]:
print("\n--- 6. Generating Submission File ---")
submission_df = test_df_processed[['id1', 'id2', 'id3', 'id5']].copy()
submission_df['pred'] = final_predictions
submission_df.to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' created successfully.")
print(submission_df.head())